# 3. 长期记忆

## 3.1 基本理解

### 3.1.1 什么是长期记忆

短期记忆记录的是 会话级别（线程，Thread） 的数据， 会话间不共享 。

而长期记忆记录的是用户特定或应用级别的数据，任何会话都可以随时访问。

比如：
1. 你喜欢简短回答
2. 你偏好 Python
3. 某个用户是 VIP

这类信息不属于某一条聊天，而属于“用户/组织/应用本身”。

### 3.1.2 类型划分

LangChain参考CoALA paper将长期记忆划分为三类：

1. 类型1：Semantic Memory（语义记忆）
即“事实类记忆”，记录事实/用户偏好/概念，如：

* 用户喜欢简洁回答
* 用户常用中文
* 某个公司属于哪个行业类型
2. Episodic Memory（情景记忆）
即“经验类记忆”，记录Agent过去执行的动作，如：
* 过去某个任务是怎么成功的
* 某种用户输入下，怎样回答效果最好

在Agent里，这常常表现为 **few-shot examples（少样本示例）**：

不直接告诉模型规则，而是给它看几个“输入 -> 输出”的例子，让它照着学类型

3. Procedural Memory（程序性记忆）

即“规则/做事方法”，如
* Agent的系统提示词
* Agent的工作流程
* 工具调用规则

### 3.1.3 存储架构
长期记忆的存储是 store -> namespace -> key -> value 的四层架构。

![长期记忆的存储架构](assets/08-long-term-store-architecture.svg)

可以类比文件系统来理解这四层：

| 层 | 类比 | 类型 | 作用 |
|---|---|---|---|
| store | 磁盘 | `BaseStore` 的子类 | 存储后端，整个应用共享一个。`InMemoryStore` 放内存（进程重启就丢），`SqliteStore` / `PostgresStore` 落盘 |
| namespace | 文件夹路径 | `tuple[str, ...]` | 给记忆分组。通常放入 `user_id` 实现用户隔离，再用最后一段区分记忆类型 |
| key | 文件名 | `str` | 在同一个 namespace 内唯一。对同一个 namespace + key 再 `put` 一次就是覆盖 |
| value | 文件内容 | `dict` | 记忆本身，必须能 JSON 序列化；字段可用 `filter` 过滤 |

namespace 设计成元组而不是一个字符串，是为了支持**按前缀检索**：

* `get` / `put` 要给完整的 namespace + key，精确定位一条记忆；
* `search` 只给 namespace 前缀，例如 `search(("users", "u_001"))` 会返回 `("users", "u_001", "profile")` 和 `("users", "u_001", "episodes")` 下的全部条目。

底层实现其实就是“两级定位”：`InMemoryStore` 内部是 `dict[namespace, dict[key, Item]]` 两层字典；`SqliteStore` / `PostgresStore` 是一张主键为 `(prefix, key)` 的表，`prefix` 就是 namespace 用点号拼成的字符串，如 `"users.u_001.profile"`。